# Decentralized Swarm Interception with MAPPO

**A small research prototype inspired by the ENSTA U2IS internship proposal.**

This notebook focuses on the learned target-assignment component: two interceptors coordinate against three scripted intruders. A shared decentralized actor selects a target from local observations; a centralized critic is used only during training (CTDE). A deterministic local pursuit controller translates the chosen target into a movement command.

> **Scope and honesty:** movement guidance is not learned in this version. The simulator uses simplified 2D kinematics, not drone flight dynamics. The intruders are scripted; no learned intruder team or self-play is implemented. Results are specific to this toy configuration.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.swarm_env import EnvConfig, SwarmInterceptionEnv
from src.mappo import MAPPOAgent, MAPPOConfig, train_mappo
from src.evaluation import evaluate_agent, evaluate_baseline, assignment_baseline_actions
from src.mappo import decentralized_pursuit_guidance

torch.set_num_threads(1)
SEED = 7
OUTPUT = Path('results')
OUTPUT.mkdir(exist_ok=True)

env = SwarmInterceptionEnv(EnvConfig(seed=SEED))
agent = MAPPOAgent(
    obs_dim=env.obs_dim,
    state_dim=env.state_dim,
    n_targets=env.n_target_actions,
    config=MAPPOConfig(seed=SEED),
)
print(f'Agents: {env.n_interceptors} | Intruders: {env.n_intruders}')
print(f'Local observation dimension: {env.obs_dim} | Central state dimension: {env.state_dim}')

## 1. Train the decentralized MAPPO assignment policy

Each actor receives only its local observation and teammate's last target-intention message. The centralized critic sees the global state during training. Both interceptors share actor parameters and receive the same team reward.

In [ ]:
history, update_history = train_mappo(
    env,
    agent,
    total_episodes=512,
    rollout_episodes=16,
    seed_start=SEED * 1000,
    log_every=32,
)
torch.save(agent.model.state_dict(), OUTPUT / 'mappo_actor_critic.pt')
rewards = np.asarray([episode['episode_reward'] for episode in history])
print(f'Training episodes: {len(history)}')
print(f'Final 30-episode mean reward: {rewards[-30:].mean():.3f}')

In [ ]:
window = 30
moving_average = np.convolve(rewards, np.ones(window) / window, mode='valid')
plt.figure(figsize=(8, 4))
plt.plot(rewards, alpha=0.25, label='Episode reward')
plt.plot(np.arange(window - 1, window - 1 + len(moving_average)), moving_average,
         label=f'Moving average ({window})')
plt.xlabel('Training episode')
plt.ylabel('Team reward')
plt.title('MAPPO training curve')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT / 'training_curve.png', dpi=180)
plt.show()

## 2. Evaluate on held-out reset seeds

The learned policy and the baseline are tested on the same 1,000 reset seeds. The baseline recomputes a minimum-total-distance one-to-one assignment at each step and uses the same pursuit controller.

In [ ]:
EVAL_EPISODES = 1000
EVAL_SEED_START = 900000
learned_metrics = evaluate_agent(env, agent, episodes=EVAL_EPISODES, seed_start=EVAL_SEED_START)
baseline_metrics = evaluate_baseline(env, episodes=EVAL_EPISODES, seed_start=EVAL_SEED_START)

comparison = pd.DataFrame(
    [learned_metrics, baseline_metrics],
    index=['MAPPO learned assignment', 'Distance-only assignment baseline'],
)
comparison.T

In [ ]:
metrics = {
    'learned_mappo': learned_metrics,
    'assignment_baseline': baseline_metrics,
    'training_episodes': 512,
    'seed': SEED,
    'evaluation_seed_start': EVAL_SEED_START,
    'evaluation_seed_end': EVAL_SEED_START + EVAL_EPISODES - 1,
    'note': ('Toy 2D prototype. MAPPO learns decentralized target selection only; '
             'low-level movement uses a deterministic local pursuit controller. '
             'Results are specific to this fixed configuration.')
}
(OUTPUT / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

In [ ]:
labels = ['Full-defense rate', 'Intruder interception rate']
map_values = [learned_metrics['full_defense_rate'], learned_metrics['intruder_interception_rate']]
base_values = [baseline_metrics['full_defense_rate'], baseline_metrics['intruder_interception_rate']]
x = np.arange(len(labels))
width = 0.34
plt.figure(figsize=(7.5, 4))
plt.bar(x - width / 2, map_values, width, label='MAPPO')
plt.bar(x + width / 2, base_values, width, label='Distance-only assignment')
plt.xticks(x, labels)
plt.ylim(0, 1.05)
plt.ylabel('Rate')
plt.title(f'Held-out evaluation ({EVAL_EPISODES:,} reset seeds)')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT / 'evaluation_comparison.png', dpi=180)
plt.show()

## 3. Qualitative episode visualization

The next cell draws one held-out rollout for each method. Paths show the two interceptors and three intruders. The protected asset is at the center of the arena.

In [ ]:
def collect_paths(method, seed=900021):
    obs, state, masks = env.reset(seed=seed)
    interceptor_paths = [ [p.copy()] for p in env.interceptor_pos ]
    intruder_paths = [ [p.copy()] for p in env.intruder_pos ]
    while True:
        if method == 'mappo':
            targets, _, _ = agent.act(obs, state, masks, deterministic=True)
            moves = decentralized_pursuit_guidance(obs, targets, env.n_interceptors, env.n_intruders)
        else:
            targets, moves = assignment_baseline_actions(env)
        obs, state, masks, _, done, _ = env.step(targets, moves)
        for i, p in enumerate(env.interceptor_pos):
            interceptor_paths[i].append(p.copy())
        for j, p in enumerate(env.intruder_pos):
            intruder_paths[j].append(p.copy())
        if done:
            break
    return np.asarray(interceptor_paths), np.asarray(intruder_paths)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, method, title in zip(axes, ['mappo', 'baseline'], ['MAPPO learned assignment', 'Distance-only assignment']):
    ipaths, tpaths = collect_paths(method)
    ax.add_patch(plt.Circle((0.5, 0.5), env.config.protected_radius, color='green', alpha=0.2))
    ax.scatter([0.5], [0.5], marker='*', s=170, label='Protected asset')
    for i, path in enumerate(ipaths):
        ax.plot(path[:, 0], path[:, 1], marker='.', markersize=2, label=f'Interceptor {i}')
        ax.scatter(path[0, 0], path[0, 1], marker='^', s=55)
    for j, path in enumerate(tpaths):
        ax.plot(path[:, 0], path[:, 1], linestyle='--', marker='x', markersize=2, label=f'Intruder {j}')
    ax.set(xlim=(0, 1), ylim=(0, 1), title=title, xlabel='x', ylabel='y')
    ax.set_aspect('equal')
    ax.grid(alpha=0.25)
axes[0].legend(fontsize=7, loc='upper right')
plt.tight_layout()
plt.savefig(OUTPUT / 'trajectory_comparison.png', dpi=180)
plt.show()

## 4. Interpretation and limitations

In this test configuration, the MAPPO assignment policy achieved a higher interception rate than the instantaneous distance-only baseline, but its mean time to capture was longer. The result should be read as a proof-of-concept for learned assignment—not as a realistic drone-swarm benchmark.

Next steps toward the internship proposal: (1) attention/pointer-based variable-size target selection; (2) learned low-level guidance; (3) scaling the team sizes including `N < M`; (4) vectorized simulation/JAX; (5) learned intruder policies and competitive self-play; (6) higher-fidelity validation before discussing physical systems.